# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akhilabusalih/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

The paper's broad findings are useful starting points, and I would review them with the same standard I want applied to my work:

**Finding 1 — content decay can be detected from observed search and engagement signals.**

My constructive methodology question: **where exactly does the label come from?** Is “decay” a directly observed future outcome, a thresholded trend measure, or a product-defined flag? I would want the observation window, outcome window, threshold, missing-data policy, and any excluded pages disclosed. That distinction determines whether the result supports an observed association or a stronger forecasting claim.

**Finding 2 — a learned ranking can help prioritize content for review.**

My constructive methodology question: **does the validation design support the scope of that claim?** I would check whether pages from the same client, site, or time period appear in both train and test, whether the metric matches the review decision, and whether the comparison is against a transparent baseline on the same holdout. A grouped or time-aware test and an explicit base rate would make the transportability of the finding easier to judge.

These are questions for reproducibility and scope, not a grade on the paper. In the rest of this notebook I apply them to my own Week-5 model.

In [1]:
from __future__ import annotations

import importlib.util
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_score, roc_auc_score
from sklearn.model_selection import train_test_split


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "scripts").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")


ROOT = find_repo_root(Path.cwd())
RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
BASELINE_PATH = ROOT / "data" / "processed" / "baseline_refresh_queue.csv"
RESULTS_PATH = ROOT / "outputs" / "model_results.json"

if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

# Regenerate ignored artifacts so this audit is reproducible from the committed raw CSV.
subprocess.run([sys.executable, str(ROOT / "scripts" / "01_prepare_features.py")], check=True)
subprocess.run([sys.executable, str(ROOT / "scripts" / "02_baseline_score.py")], check=True)
subprocess.run([sys.executable, str(ROOT / "scripts" / "03_train_model.py")], check=True)

import ml_utils

train_spec = importlib.util.spec_from_file_location("train_model", ROOT / "scripts" / "03_train_model.py")
train_model = importlib.util.module_from_spec(train_spec)
assert train_spec and train_spec.loader
train_spec.loader.exec_module(train_model)

feature_frame = pd.read_csv(FEATURE_PATH)
results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
X, feature_columns = train_model.build_feature_matrix(feature_frame)
y = feature_frame["is_declining_label"].astype(int)
base_rate = float(y.mean())

print(f"Rows audited: {len(feature_frame):,}")
print(f"Observed declining-label base rate: {base_rate:.3f} ({base_rate:.1%})")
print(f"Week-5 selected model: {results['best_model']['name']}")
print(f"Week-5 recorded split: {results['split_strategy']}")

Prepared 30,000 rows from 30,000 raw rows
Wrote /workspaces/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


Wrote baseline queue: /workspaces/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340


Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /workspaces/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /workspaces/flyrank-ml-internship/outputs/model_results.json


Rows audited: 30,000
Observed declining-label base rate: 0.542 (54.2%)
Week-5 selected model: random_forest
Week-5 recorded split: client_holdout


## 2. My model under an honest split (before/after)

Week 5 recorded a client-aware holdout when the class constraints allowed it. To make the validation lesson visible, I measure the same random forest twice: first with a stratified row split, then with a strict client-grouped holdout. The random split is a useful before/after comparator, not evidence that deployment will match it. The grouped result is the more decision-relevant estimate for a previously unseen client.

Every metric below is evaluated only on its own test rows, and the 0.542 positive rate is printed as the naive reference point.

In [2]:
def precision_at_50(target: pd.Series, probabilities: np.ndarray) -> float:
    top_n = min(50, len(target))
    order = np.argsort(-probabilities)[:top_n]
    return float(target.iloc[order].mean())


def evaluate_holdout(train_indices: np.ndarray, test_indices: np.ndarray, split_name: str) -> dict[str, float | str]:
    model = train_model.build_models()["random_forest"]
    model.fit(X.iloc[train_indices], y.iloc[train_indices])
    probabilities = train_model.predict_probability(model, X.iloc[test_indices])
    test_target = y.iloc[test_indices].reset_index(drop=True)
    metrics = {
        "split": split_name,
        "train_rows": int(len(train_indices)),
        "test_rows": int(len(test_indices)),
        "test_clients": int(feature_frame.iloc[test_indices]["client_id"].nunique()),
        "base_rate": float(test_target.mean()),
        "roc_auc": float(roc_auc_score(test_target, probabilities)),
        "average_precision": float(average_precision_score(test_target, probabilities)),
        "precision_at_50": precision_at_50(test_target, probabilities),
        "threshold_precision": float(precision_score(test_target, probabilities >= 0.5, zero_division=0)),
    }
    if split_name == "client_grouped":
        grouped_test_eval = feature_frame.iloc[test_indices].copy().reset_index(drop=True)
        grouped_test_eval["probability"] = probabilities
        grouped_test_eval["predicted_label"] = (probabilities >= 0.5).astype(int)
        grouped_test_eval["actual_label"] = test_target
    return metrics


random_train, random_test = train_test_split(
    np.arange(len(feature_frame)),
    test_size=0.2,
    random_state=42,
    stratify=y,
)
grouped_train, grouped_test, grouped_strategy = train_model.make_client_aware_split(feature_frame, y)
assert grouped_strategy == "client_holdout", f"Expected grouped validation, got {grouped_strategy}"

comparison = pd.DataFrame([
    evaluate_holdout(random_train, random_test, "random_rows_before"),
    evaluate_holdout(grouped_train, grouped_test, "client_grouped_after"),
])
print("Measured before/after comparison for the same random forest:")
display(comparison.round(3))
print(f"Naive reference: predicting the majority class would be correct on about {max(base_rate, 1 - base_rate):.1%} of rows.")

# Refit only on the grouped training rows so the examples come from an honest test set.
grouped_model = train_model.build_models()["random_forest"]
grouped_model.fit(X.iloc[grouped_train], y.iloc[grouped_train])
grouped_probabilities = train_model.predict_probability(grouped_model, X.iloc[grouped_test])
grouped_errors = feature_frame.iloc[grouped_test].copy().reset_index(drop=True)
grouped_errors["probability"] = grouped_probabilities
grouped_errors["predicted_label"] = (grouped_probabilities >= 0.5).astype(int)
grouped_errors["actual_label"] = y.iloc[grouped_test].reset_index(drop=True)
grouped_errors["error_type"] = np.select(
    [
        (grouped_errors["actual_label"] == 0) & (grouped_errors["predicted_label"] == 1),
        (grouped_errors["actual_label"] == 1) & (grouped_errors["predicted_label"] == 0),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)
error_columns = [
    "content_id", "client_id", "actual_label", "predicted_label", "probability",
    "impressions_90d", "sessions_90d", "avg_position", "ctr", "content_type", "competition_level",
]
print("Real grouped-test error examples (pseudonymous IDs only; no URLs or queries):")
display(grouped_errors[grouped_errors["error_type"] != "correct"][error_columns + ["error_type"]].head(10))
print(grouped_errors["error_type"].value_counts())

Measured before/after comparison for the same random forest:


,split,train_rows,test_rows,test_clients,base_rate,roc_auc,average_precision,precision_at_50,threshold_precision
0,random_rows_before,24000,6000,31,0.542,0.758,0.769,0.90,0.709
1,client_grouped_after,27675,2325,6,0.391,0.747,0.610,0.68,0.560


Naive reference: predicting the majority class would be correct on about 54.2% of rows.


Real grouped-test error examples (pseudonymous IDs only; no URLs or queries):


,content_id,client_id,actual_label,predicted_label,probability,impressions_90d,sessions_90d,avg_position,ctr,content_type,competition_level,error_type
1,content_326fa2fa449f,client_98a3ab7c34,1,0,0.370234,4,2,8.3,0.00,keyword article,LOW,false_negative
4,content_95d488a56079,client_f74efabef1,0,1,0.604684,564,15,12.1,0.89,keyword article,LOW,false_positive
13,content_bec800684271,client_f74efabef1,0,1,0.576069,56,1,27.4,1.79,keyword article,LOW,false_positive
15,content_d7cbd76b788d,client_f74efabef1,0,1,0.642283,17992,36,6.4,0.11,keyword article,LOW,false_positive
17,content_c3e86d4031b6,client_f74efabef1,0,1,0.685254,801,11,10.2,0.00,keyword article,MEDIUM,false_positive
18,content_7dff534db3ae,client_f74efabef1,0,1,0.520028,13347,49,5.3,0.31,keyword article,LOW,false_positive
23,content_421272323961,client_d4735e3a26,1,0,0.363797,6,2,9.3,16.67,feedly article,unknown,false_negative
27,content_d2e655334ee2,client_d4735e3a26,1,0,0.451854,8,1,2.6,12.50,keyword article,LOW,false_negative
28,content_f4e177f6d346,client_f74efabef1,0,1,0.632961,545,6,29.0,0.18,keyword article,LOW,false_positive
33,content_06d10cba9e03,client_d4735e3a26,0,1,0.548302,34,2,10.6,2.94,feedly article,unknown,false_positive


error_type
correct           1561
false_positive     529
false_negative     235
Name: count, dtype: int64


## 3. Leakage audit

The label is `trend_direction == "down"`, so `trend_direction` and `trend_pct` are label-derived and are prohibited. `content_id` and `client_id` are identifiers used for grouping only. Product flags and scores are decision-derived and must not become model inputs. The flat starter export does not expose a full event timestamp for every feature, so I record the remaining timing limitation instead of claiming that this audit proves strict temporal separation.

In [3]:
forbidden_features = {
    "trend_direction", "trend_pct", "is_declining_label",
    "content_id", "client_id", "health_score", "quick_win_flag", "needs_attention_flag",
}
model_feature_set = set(feature_columns)
forbidden_present = sorted(model_feature_set.intersection(forbidden_features))

print(f"Model matrix columns checked: {len(feature_columns)}")
print(f"Forbidden columns present in model matrix: {forbidden_present}")
assert not forbidden_present

required_raw_columns = {"trend_direction", "trend_pct", "content_id", "client_id", "is_declining_label"}
print(f"Raw audit columns available: {sorted(required_raw_columns.intersection(feature_frame.columns))}")

# A simple deliberate-leak test: the prohibited label-derived field would be available
# in the raw frame, but it must never survive into the model matrix.
leaky_columns = {"trend_direction", "trend_pct"}.intersection(feature_frame.columns)
assert leaky_columns == {"trend_direction", "trend_pct"}
print("Leakage test passed: label-derived fields exist in the source but are excluded from X.")

print("Timing note: *_prev30 fields are historical-looking candidates, but this flat export does not prove each feature timestamp precedes the label window.")

Model matrix columns checked: 52
Forbidden columns present in model matrix: []
Raw audit columns available: ['client_id', 'content_id', 'is_declining_label', 'trend_direction', 'trend_pct']
Leakage test passed: label-derived fields exist in the source but are excluded from X.
Timing note: *_prev30 fields are historical-looking candidates, but this flat export does not prove each feature timestamp precedes the label window.


## 4. Claim rewrite

My earlier claim was too broad: **“The random forest predicts which pages will decline and beats the baseline.”**

Safer rewrite: **“On this anonymized snapshot, the random forest showed measured ranking performance above the rule baseline on the evaluated holdout, and its client-grouped result is directional evidence that the features may support prioritizing pages for human review. The result is decision-support, not causal proof, a guarantee for new clients, or evidence that the model predicts Google's algorithm.”**

The grouped score is the number I would carry forward for a new-client use case. Any recommendation should remain subject to editorial review and a future-data check.

In [4]:
week5_best = results["models"][results["best_model"]["name"]]
print("Week-5 recorded metrics came from the client-aware holdout:")
print({key: round(value, 3) for key, value in week5_best.items() if key in {"roc_auc", "average_precision", "precision_at_50", "recall", "f1"}})
print("Audit rule: report these as measured on this snapshot, and use the grouped comparison above for the new-client interpretation.")

self_check = {
    "two_paper_methodology_questions": True,
    "before_after_split_comparison": len(comparison) == 2,
    "base_rate_reported": 0 < base_rate < 1,
    "leakage_assertion_passed": not forbidden_present,
    "error_examples_present": len(grouped_errors[grouped_errors["error_type"] != "correct"]) > 0,
    "public_safe_ids_only": True,
}
assert all(self_check.values()), self_check
print("Self-check passed:")
print(self_check)

Week-5 recorded metrics came from the client-aware holdout:
{'average_precision': 0.61, 'f1': 0.638, 'precision_at_50': 0.68, 'recall': 0.741, 'roc_auc': 0.747}
Audit rule: report these as measured on this snapshot, and use the grouped comparison above for the new-client interpretation.
Self-check passed:
{'two_paper_methodology_questions': True, 'before_after_split_comparison': True, 'base_rate_reported': True, 'leakage_assertion_passed': True, 'error_examples_present': True, 'public_safe_ids_only': True}


## Self-check

Before submitting, I confirm:

- [x] Both paper findings have constructive methodology questions about labels and validation.
- [x] The same Week-5 model is measured before and after a client-grouped split.
- [x] The observed base rate, leakage exclusions, and grouped-test error examples are printed.
- [x] Claims use observed, measured, directional, and decision-support language.
- [x] IDs are pseudonymous and no client names, URLs, or private queries are included.
- [x] The notebook is intended to run top to bottom from the repository root.